# Recompute integration metrics across trained models

Papermill-parameterized notebook that loads the shared adata once, then iterates over
a list of trained model results folders, loading each model's latent representation
and leiden clusters from saved CSVs, and computing integration metrics.

**Exclusions applied**:
- `pbmc_tea_seq` dataset removed from all computations (labels look randomized)
- `Platelet` in `level_1` replaced with NaN (not a cell type)

**Metrics computed**:
1. Custom sklearn-based metrics (`compute_integration_metrics`): silhouette, ARI, NMI, LISI
2. scib-metrics `Benchmarker`: bio conservation + batch correction suite with signature plot

In [ ]:
# Papermill parameters
results_base = "/nfs/team205/vk7/sanger_projects/my_packages/regularizedvi/results"
experiment_names = [
    "immune_integration_rna_embryo_es2_no_tea",
    "immune_integration_rna_embryo_es2_no_tea_small",
    "immune_integration_rna_embryo_es2_no_tea_disp1",
    "immune_integration_rna_embryo_es2_no_tea_small_disp1",
    "immune_integration_rna_embryo_es2_no_tea_disp2",
    "immune_integration_rna_embryo_es2_no_tea_small_disp2",
    "immune_integration_rna_embryo_es2_no_tea_dwl2_1",
    "immune_integration_rna_embryo_es2_no_tea_small_dwl2_1",
    "immune_integration_rna_embryo_es2_no_tea_disp1_dwl2_1",
    "immune_integration_rna_embryo_es2_no_tea_small_disp1_dwl2_1",
    "immune_integration_rna_embryo_es2_no_tea_disp2_dwl2_1",
    "immune_integration_rna_embryo_es2_no_tea_small_disp2_dwl2_1",
]
output_folder = (
    "/nfs/team205/vk7/sanger_projects/my_packages/regularizedvi/results/immune_integration_metrics_comparison"
)
adata_path = "/nfs/team205/vk7/sanger_projects/my_packages/regularizedvi/results/immune_integration/adata_rna.h5ad"
subsample_n = 50000
random_state = 42
n_jobs = 4
exclude_datasets = "pbmc_tea_seq"
exclude_labels = "Platelet"

In [ ]:
import os
import time

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

results_folders = [os.path.join(results_base, name) for name in experiment_names]
os.makedirs(output_folder, exist_ok=True)
print(f"Models to evaluate: {len(results_folders)}")
print(f"Output folder: {output_folder}")

In [ ]:
print(f"Loading adata from {adata_path}...")
t0 = time.time()
adata = sc.read_h5ad(adata_path)
print(f"Loaded: {adata.shape} in {time.time() - t0:.1f}s")

In [ ]:
# Apply exclusions
n_before = adata.n_obs

# Exclude datasets
if exclude_datasets:
    _excl = [s.strip() for s in exclude_datasets.split(",")]
    adata = adata[~adata.obs["dataset"].isin(_excl)].copy()
    print(f"Excluded datasets {_excl}: {n_before} -> {adata.n_obs} cells")

# Exclude labels (set to NaN so metrics filter them out)
if exclude_labels:
    _excl_labels = [s.strip() for s in exclude_labels.split(",")]
    for lab in _excl_labels:
        n_match = (adata.obs["level_1"] == lab).sum()
        adata.obs.loc[adata.obs["level_1"] == lab, "level_1"] = np.nan
        print(f"Excluded label '{lab}': {n_match} cells set to NaN")

# Verify labelled cell count
has_label = adata.obs["level_1"].notna() & (adata.obs["level_1"] != "") & (adata.obs["level_1"].astype(str) != "nan")
print(f"\nFiltered adata: {adata.n_obs} cells, {has_label.sum()} labelled")
print(f"Datasets: {sorted(adata.obs['dataset'].unique().tolist())}")
print(f"Labels: {sorted(adata.obs.loc[has_label, 'level_1'].unique().tolist())}")

all_obs_names = set(adata.obs_names)

## 1. Custom integration metrics (sklearn-based)

In [ ]:
from regularizedvi.plt import compute_integration_metrics

custom_metrics = {}

for results_folder in results_folders:
    model_name = os.path.basename(results_folder)
    outputs_dir = os.path.join(results_folder, "model", "outputs")

    scvi_path = os.path.join(outputs_dir, "X_scVI.csv")
    leiden_path = os.path.join(outputs_dir, "leiden_k50.csv")

    if not os.path.exists(scvi_path):
        print(f"SKIP {model_name}: {scvi_path} not found")
        continue

    print(f"\n{'=' * 60}")
    print(f"Processing: {model_name}")
    print(f"{'=' * 60}")

    t0 = time.time()
    X_scVI_df = pd.read_csv(scvi_path, index_col=0)
    leiden_df = pd.read_csv(leiden_path, index_col=0)
    print(f"  Loaded CSVs: X_scVI {X_scVI_df.shape}, leiden {leiden_df.shape} ({time.time() - t0:.1f}s)")

    # Intersect with filtered adata
    common_obs = sorted(set(X_scVI_df.index) & all_obs_names)
    print(f"  Model cells: {len(X_scVI_df)}, adata cells: {adata.n_obs}, intersection: {len(common_obs)}")

    if len(common_obs) < 1000:
        print(f"  SKIP: too few overlapping cells ({len(common_obs)})")
        continue

    # Subset adata and inject model outputs
    adata_sub = adata[common_obs].copy()
    adata_sub.obsm["X_scVI"] = X_scVI_df.loc[common_obs].values
    adata_sub.obs["leiden"] = leiden_df.loc[common_obs, "leiden"].astype(str).values

    t0 = time.time()
    metrics_df = compute_integration_metrics(
        adata_sub,
        latent_key="X_scVI",
        label_key="level_1",
        batch_key="dataset",
        leiden_key="leiden",
        subsample_n=subsample_n,
        random_state=random_state,
    )
    print(f"  Metrics computed in {time.time() - t0:.1f}s")

    metrics_df["model"] = model_name
    custom_metrics[model_name] = metrics_df

    del adata_sub, X_scVI_df, leiden_df

print(f"\nCompleted {len(custom_metrics)} / {len(results_folders)} models")

In [ ]:
# Save custom metrics
if custom_metrics:
    combined_long = pd.concat(custom_metrics.values(), ignore_index=True)
    long_path = os.path.join(output_folder, "custom_metrics_long.csv")
    combined_long.to_csv(long_path, index=False)
    print(f"Saved: {long_path}")

    # Wide format for comparison
    numeric_mask = combined_long["value"].apply(lambda x: isinstance(x, (int, float, np.floating)))
    custom_wide = combined_long[numeric_mask].pivot_table(
        index="metric", columns="model", values="value", aggfunc="first"
    )
    wide_path = os.path.join(output_folder, "custom_metrics_wide.csv")
    custom_wide.to_csv(wide_path)
    print(f"Saved: {wide_path}")

    # Key metrics summary
    key_metrics = [
        "silhouette_label",
        "silhouette_batch",
        "ARI_leiden_vs_label",
        "NMI_leiden_vs_label",
        "iLISI_median",
        "cLISI_median",
    ]
    summary = custom_wide.loc[custom_wide.index.isin(key_metrics)]
    print("\n=== Key Custom Metrics ===")
    display(summary.round(4))

## 2. scib-metrics Benchmarker

In [ ]:
from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection

# Load all embeddings into a single adata for Benchmarker comparison.
# Use the intersection of obs_names across all models.
model_obs_sets = {}
for results_folder in results_folders:
    model_name = os.path.basename(results_folder)
    scvi_path = os.path.join(results_folder, "model", "outputs", "X_scVI.csv")
    if not os.path.exists(scvi_path):
        continue
    idx = pd.read_csv(scvi_path, usecols=[0]).iloc[:, 0].tolist()
    model_obs_sets[model_name] = set(idx)

# Common cells across all models AND filtered adata
common_all = all_obs_names.copy()
for obs_set in model_obs_sets.values():
    common_all &= obs_set
common_all = sorted(common_all)
print(f"Common cells across all {len(model_obs_sets)} models: {len(common_all)}")

# Build adata with all embeddings
adata_bench = adata[common_all].copy()

embedding_keys = []
for results_folder in results_folders:
    model_name = os.path.basename(results_folder)
    scvi_path = os.path.join(results_folder, "model", "outputs", "X_scVI.csv")
    if not os.path.exists(scvi_path):
        continue

    X_scVI_df = pd.read_csv(scvi_path, index_col=0)
    obsm_key = f"X_{model_name}"
    adata_bench.obsm[obsm_key] = X_scVI_df.loc[common_all].values
    embedding_keys.append(obsm_key)
    print(f"  Added embedding: {obsm_key} {adata_bench.obsm[obsm_key].shape}")

print(f"\n{len(embedding_keys)} embeddings loaded into adata_bench ({adata_bench.n_obs} cells)")

In [ ]:
# Filter to labelled cells for scib Benchmarker
label_key = "level_1"
batch_key = "dataset"

mask = (
    adata_bench.obs[label_key].notna()
    & (adata_bench.obs[label_key] != "")
    & (adata_bench.obs[label_key].astype(str) != "nan")
)
adata_eval = adata_bench[mask].copy()
print(f"Labelled cells for scib: {adata_eval.n_obs}")

# Subsample if too large (pynndescent OOMs on 100k+ cells × 12 embeddings)
scib_max_cells = 50000
if adata_eval.n_obs > scib_max_cells:
    rng = np.random.RandomState(random_state)
    # Stratified subsample by label + batch
    strat = adata_eval.obs[label_key].astype(str) + "___" + adata_eval.obs[batch_key].astype(str)
    unique_strats, counts = np.unique(strat.values, return_counts=True)
    sample_idx = []
    for s, c in zip(unique_strats, counts, strict=False):
        s_idx = np.where(strat.values == s)[0]
        n_take = max(1, int(round(scib_max_cells * c / len(strat))))
        n_take = min(n_take, len(s_idx))
        sample_idx.extend(rng.choice(s_idx, size=n_take, replace=False))
    sample_idx = np.array(sample_idx[:scib_max_cells])
    adata_eval = adata_eval[sample_idx].copy()
    print(f"Subsampled to {adata_eval.n_obs} cells for scib Benchmarker")

print(f"  Labels: {adata_eval.obs[label_key].nunique()}")
print(f"  Batches: {adata_eval.obs[batch_key].nunique()}")

# Run Benchmarker
t0 = time.time()
bm = Benchmarker(
    adata_eval,
    batch_key=batch_key,
    label_key=label_key,
    embedding_obsm_keys=embedding_keys,
    pre_integrated_embedding_obsm_key=embedding_keys[0],
    bio_conservation_metrics=BioConservation(
        isolated_labels=True,
        nmi_ari_cluster_labels_kmeans=True,
        silhouette_label=True,
        clisi_knn=True,
    ),
    batch_correction_metrics=BatchCorrection(
        ilisi_knn=True,
        kbet_per_label=True,
        graph_connectivity=True,
        pcr_comparison=True,
    ),
    n_jobs=n_jobs,
)
bm.benchmark()
print(f"scib Benchmarker completed in {time.time() - t0:.1f}s")

In [ ]:
# Get results and clean up model names
scib_df = bm.get_results(min_max_scale=False, clean_names=True)

# Replace obsm key names with short experiment names
prefix = "X_immune_integration_rna_embryo_es2_"
scib_df.index = [idx.replace(prefix, "") if isinstance(idx, str) else idx for idx in scib_df.index]

# Save
scib_path = os.path.join(output_folder, "scib_metrics.csv")
scib_df.to_csv(scib_path)
print(f"Saved: {scib_path}")
display(scib_df)

## 3. scib signature heatmap

In [ ]:
# scib signature heatmap (adapted from cell2state_embryo benchmark)
# Drop 'Metric Type' row if present
metric_type_mask = scib_df.index == "Metric Type"
plot_df = scib_df[~metric_type_mask].copy()

# Select numeric columns only
numeric_cols = plot_df.select_dtypes(include=[np.number]).columns.tolist()
plot_df = plot_df[numeric_cols]

fig, ax = plt.subplots(figsize=(max(14, len(numeric_cols) * 1.2), max(4, len(plot_df) * 0.5)))
im = ax.imshow(plot_df.values.astype(float), aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)

ax.set_xticks(range(len(plot_df.columns)))
ax.set_xticklabels(plot_df.columns, rotation=45, ha="right", fontsize=9)
ax.set_yticks(range(len(plot_df)))
ax.set_yticklabels(plot_df.index, fontsize=9)

# Annotate cells with values
for i in range(len(plot_df)):
    for j in range(len(plot_df.columns)):
        val = plot_df.iloc[i, j]
        if pd.notna(val):
            color = "white" if val < 0.3 or val > 0.7 else "black"
            ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=7, color=color)

ax.set_title("scib-metrics integration benchmark", fontsize=12)
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()

# Save
for ext in ["svg", "png"]:
    fig.savefig(os.path.join(output_folder, f"scib_signature.{ext}"), dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved signature plot to {output_folder}/scib_signature.{{svg,png}}")